# EXP-2026-007 / Q5-D — order-preserving beat identity join (quest54)

## `IMPLEMENTED — FULL RESULT NOT RUN`
## `EXP-2026-007 SCIENTIFIC RESULT NOT RUN`

### 이 notebook 의 상태 — 먼저 읽을 것

- **구현 승인과 실제 실행 승인은 별개의 승인이다.**
  사용자는 **구현(코드 작성)만 승인**했다. 등록된 실데이터(MIT-BIH `.atr` ·
  `mamba_data.npz` · V9/V10 캐시 · result NPZ)에서 beat join 을 **실행하는 승인은
  아직 없다**. 명세 「Status boundary」 4항이 두 번째 승인을 따로 요구한다.
- **이 notebook 에는 실제 결과가 없다.** 미실행 상태로 커밋됐고 출력 셀이 전부
  비어 있다. 아래 어떤 표·그래프도 지금은 값을 담고 있지 않으며, 실행 승인 이후
  run bundle 이 생겼을 때 **그 파일에서 읽어** 채워진다.
- 숫자를 셀 안에서 다시 계산해 적어 넣지 않는다. 결과 셀은 전부
  `decision.json` · `null_summary.json` · `bootstrap.json` · `*.csv` 를 읽는다.
  **낡은 노트북으로 결과를 추측하지 않는다**(CLAUDE.md 「실행 로그 루프」).

### 판정 후보

`JOIN_INPUT_ABSENT` · `JOIN_RULE_FALSIFIED` · `JOIN_SELECTION_BIASED` ·
`JOIN_UNRESOLVED` · `JOIN_IDENTIFIABLE` — 그리고 지금 상태인 `JOIN_RESULT_NOT_RUN`.

### 봉인된 것 (이 substage 내내)

V10 probability 값 · DS2 association 통계 · S PR-AUC · 모델 학습/재학습 ·
P-wave delineation 재실행 · Drive 변경. join 이 자체 gate 를 통과해도 association
은 **또 다른 별도 승인** 전까지 열리지 않는다.

### 목차

1. 환경 및 승인 gate · 2. 입력 asset/hash 확인 · 3. 44-record ledger 표 ·
4. synthetic fixture 결과 · 5. Leg 1 replay audit · 6. Leg 2 record-wise join ·
7. DS1 gate report · 8. DS2 frozen gate report · 9. negative-control/null plots ·
10. class·record coverage plots · 11. ambiguous/unmatched 원인표 ·
12. equal-count 36 vs mismatch 8 진단 비교 · 13. record 105·111·116·208·222 상세표 ·
14. record 232 S-share 원분포 대 certification 후 inflation ·
15. decision tree 최종 판정 · 16. 사람이 읽을 수 있는 해석 요약 ·
17. Drive bundle 저장 및 ingest 단계

## 1. 환경 및 승인 gate

이 셀들은 **아무 등록 자산도 열지 않는다.** `DESIGN` 과 `SYNTHETIC_FIXTURES` 는
합성 데이터만 쓰고, `JOIN_REPORT` 는 이미 만들어진 bundle 을 다시 읽을 뿐이다.
나머지 네 mode 는 모듈이 **파일을 열기 전에** 별도 실행 승인을 요구하며 중단한다.

In [ ]:
# ── 셀 1: 실행 설정 (정확히 하나의 mode) ─────────────────────────────────────
VALID_MODES = ("DESIGN", "SYNTHETIC_FIXTURES", "LEG1_REPLAY_AUDIT",
               "LEG2_RECORD_JOIN", "DS1_GATE", "DS2_GATE", "JOIN_REPORT")
MODE = "DESIGN"          # 실행 승인 전에는 DESIGN / SYNTHETIC_FIXTURES / JOIN_REPORT 만
assert MODE in VALID_MODES, f"MODE must be one of {VALID_MODES}"

# 병합 후에는 "main" 으로 바꾼다.
BRANCH = "claude/q5d-recordwise-beat-join-implementation"
NEED_TESTS = 300          # 회귀 테스트 최소 통과 수

# 실행 승인이 생긴 뒤에야 채운다. 지금은 비워 둔다 — 값이 없으면 모듈이 멈춘다.
RUN_DIR = ""              # 예: /content/drive/MyDrive/MedKOS/ecg-model/runs/<ts>_EXP-2026-007_q5d_beat_join
print("MODE =", MODE)
print("이 notebook 은 미실행 상태로 커밋됐다: 아래 결과 셀은 bundle 파일에서 읽는다.")

In [ ]:
# ── 셀 2: repo 준비 + commit SHA + 회귀 테스트 ───────────────────────────────
import os, subprocess, sys

BRANCH = globals().get("BRANCH", "main")
MODE = globals().get("MODE", "DESIGN")
NEED_TESTS = int(globals().get("NEED_TESTS", 300))

REPO = "/content/my-github-test"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "50",
                    "https://github.com/ehdbddl06001-ui/my-github-test.git",
                    REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("commit:", COMMIT)

sys.path.insert(0, os.path.join(REPO, "mit-bih"))
import q5d_order_preserving_beat_join as BJ
print("module version:", BJ.MODULE_VERSION, BJ.MODULE_BUILD)
print("rule fingerprint:", BJ.rule_fingerprint())

# 회귀 테스트를 먼저 돌린다. 여기서 깨지면 아래는 볼 필요가 없다.
proc = subprocess.run([sys.executable,
                       os.path.join(REPO, "mit-bih",
                                    "test_q5d_order_preserving_beat_join.py")],
                      capture_output=True, text=True)
tail = proc.stdout.strip().splitlines()[-1]
print(tail)
assert proc.returncode == 0, proc.stdout[-3000:]

In [ ]:
# ── 셀 3: 승인 gate — 지금 무엇이 열려 있고 무엇이 닫혀 있는가 ───────────────
MODE = globals().get("MODE", "DESIGN")

print(BJ.NO_EXECUTION_BANNER)
print()
print(BJ.APPROVAL_NOTE)
print()
print("실행 승인 상태 :", BJ.execution_is_approved(None))   # False 여야 한다
print("승인 없이 가능 :", BJ.OFFLINE_MODES)
print("별도 승인 필요 :", BJ.MODES_NEEDING_EXECUTION_APPROVAL)
print()

if MODE in BJ.MODES_NEEDING_EXECUTION_APPROVAL:
    raise SystemExit(
        f"MODE={MODE} 은 등록 자산을 연다. 사용자의 두 번째 승인(실행 승인)을 받은 뒤"
        f" 모듈의 실행 승인 인자를 넘겨야 하며, 이 notebook 은 그 값을 담고 있지 않다."
        f" 지금은 {BJ.OFFLINE_MODES} 만 실행한다.")

print(BJ.design_card(MODE, execution_approved=False))

In [ ]:
# ── 셀 4: 결과 로더 — 모든 결과 셀은 오직 이 함수로만 값을 얻는다 ────────────
import csv, json, os

def bundle_path(name):
    RUN_DIR = globals().get("RUN_DIR", "")
    assert RUN_DIR, ("RUN_DIR 이 비어 있다 — 아직 실행된 run bundle 이 없다.\n"
                     "이 notebook 은 IMPLEMENTED / FULL RESULT NOT RUN 상태다.")
    return os.path.join(RUN_DIR, name)

def load_json(name):
    "결과 숫자는 파일에서 읽는다. 셀 안에서 다시 계산하거나 옮겨 적지 않는다."
    with open(bundle_path(name), encoding="utf-8") as fh:
        return json.load(fh)

def load_csv(name):
    with open(bundle_path(name), encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

def have_results():
    RUN_DIR = globals().get("RUN_DIR", "")
    return bool(RUN_DIR) and os.path.exists(os.path.join(RUN_DIR, "decision.json"))

def need_results(section):
    if not have_results():
        print(f"[{section}] 결과 없음 — FULL RESULT NOT RUN.")
        print("  실행 승인 후 run bundle 이 생기면 이 셀이 그 파일에서 값을 읽는다.")
        return False
    return True

print("결과 유무:", have_results())

## 2. 입력 asset/hash 확인

`JOIN_INPUT_ABSENT` 는 **자재 계약 실패**일 때만 발화한다 — canonical mamba 자산과
source/meta hash, V9·V10 캐시와 44-record 경계, detection-order 계약,
cache→result NPZ 위치 계약. **join 성능이 나쁘다는 이유로는 이 분기에 오지 않는다.**

실행 승인 전이므로 여기서는 **무엇을 대조할지**만 보여 준다. 실제 hash 대조는
`manifest.json` 이 생긴 뒤 그 파일에서 읽는다.

In [ ]:
# ── 셀 5: 입력 계약 — 지금은 목록만, 실행 후에는 manifest.json 에서 읽는다 ───
print("이 join 이 요구하는 자재 계약:")
for line in (
    "canonical mamba 자산 + source/meta hash + 44-record 순서·개수·단위·drop 의미",
    "V9·V10 cache/meta 자산 + 44 record 경계 + detection-order 계약 + 두 ledger 동일성",
    "cache row -> result NPZ row 위치 계약 (pid 블록 길이 = 등록 cache 경계)",
    "등록 canonical 자산과 byte 단위로 연결되지 않는 중복본은 사용 불가",
):
    print("  -", line)
print()
print("읽는 result NPZ 키:", BJ.RESULT_NPZ_DS1_AUDIT_KEYS,
      "· 봉인:", BJ.RESULT_NPZ_SEALED)
print("`t` 는 join key 로 쓸 수 없다 (record 마다 0에서 재시작 + 필터링된 RR 누적).")

if need_results("2. 입력 asset/hash"):
    manifest = load_json("manifest.json")
    print(json.dumps(manifest.get("inputs", {}), indent=2, ensure_ascii=False))
    print("ledger 검증:", manifest["ledger"]["ok"])
    print("code guard :", manifest["code"]["clean"], manifest["code"]["sha256"][:16])

## 3. 44-record ledger 표

record 경계는 **등록된 대장에서 산술로** 잘린다. 라벨이나 join 품질로 추론하지
않는다. 일치 36 record 와 불일치 8 record 는 **사전 등록된 보고 층(strata)** 이고,
둘 다 **같은 matcher** 를 통과한다. 개수가 같다고 위치 동일성으로 간주하지 않는다.

In [ ]:
# ── 셀 6: 44-record ledger (등록 상수에서 생성, 결과와 무관) ─────────────────
report = BJ.verify_ledger()
print("ledger ok        :", report["ok"], report["problems"])
print("records          :", report["records"],
      f"(equal {report['equal_count_records']} · mismatch {report['mismatched_records']})")
print("cache / mamba    :", report["cache_total"], "/", report["mamba_total"],
      "· difference", report["total_difference"])
print()
print(BJ.ledger_table())

## 4. synthetic fixture 결과

합성 fixture 는 **DS1 을 보기 전에** 돈다. false certified pair 가 하나라도 나오면
`JOIN_RULE_FALSIFIED` 로 즉시 종결한다. 이 절은 실행 승인 없이도 돌릴 수 있다 —
등록 자산을 하나도 열지 않기 때문이다.

In [ ]:
# ── 셀 7: fixture 배터리 (합성 데이터만) ────────────────────────────────────
outcomes = BJ.run_synthetic_fixtures()
print(BJ.fixture_card(outcomes))
assert BJ.fixtures_passed(outcomes), "false certified pair 가 있으면 여기서 멈춘다"

## 5. Leg 1 replay audit — `.atr` → mamba

결정론적 **source replay** 다. 통계적 join 이 아니다. 등록된 N/S/V symbol map,
annotation 위치 `pos` 기준 150-sample 경계 규칙, 5개 미만 record 규칙 셋을 원
`.atr` 만으로 재계산하고, record별 개수·순서·RR 을 커밋된 계보 대장과 대조한다.

**첫·끝 beat 는 eligible 하다.** 첫 pre-RR 은 첫 interval 의 복제이고 마지막
post-RR 은 마지막 interval 의 복제다 — 없는 것이 아니다.

불일치는 `JOIN_RULE_FALSIFIED` + `failed_leg = LEG1_SOURCE_REPLAY` 이고 **Leg 2 는
시작하지 않는다.**

In [ ]:
# ── 셀 8: Leg 1 audit — 실행 승인 후 decision.json/log 에서 읽는다 ──────────
print("Leg 1 규칙 (등록 상수):")
print("  symbol map :", {k: v for k, v in sorted(BJ.AAMI_SYMBOL_MAP.items())})
print("  경계 규칙  :", f"{BJ.WIN_BEFORE} <= pos < len(signal) - {BJ.WIN_AFTER}")
print("  record 규칙:", f"유효 beat < {BJ.MIN_VALID_BEATS} 이면 record 통째 제외")
print("  RR         : 필터링 이후, 초 단위, 첫·끝 복제 → 첫·끝 beat eligible")
print("  기대 총계  : DS1", BJ.REGISTERED_MAMBA_TOTALS["DS1"],
      "· DS2", BJ.REGISTERED_MAMBA_TOTALS["DS2"])

if need_results("5. Leg 1 replay audit"):
    decision = load_json("decision.json")
    leg1 = [g for g in decision["gates"] if g["gate"].startswith("2a")]
    for gate in leg1:
        print(f"  {gate['gate']}: passed={gate['passed']} · {gate['detail']}")
    print("failed_leg:", decision["failed_leg"])

## 6. Leg 2 record-wise join — mamba → V9/V10 위치 행

검출기 의존이라 `.atr` 로 재계산되지 않는다. 대신 등록 캐시가 행을 물질화해
두었고, record 경계는 대장에서 산술로 나온다.

- V9/V10 행 순서 = `detect_r()` **검출 순서**. `.atr` ordinal 이 아니다.
- result NPZ 는 `prob`·`y`·`pid` 만 저장한다 → identity 는 **위치뿐**이다.
- **전역 정렬 금지.** DS2 의 105·111·222 결손이 이후 모든 record 를 밀어 버린다.
- gap 은 양쪽에서 허용하되 **어떤 행도 impute 하지 않는다.**
- 후보 간선: `|Δpre| <= 1` **그리고** `|Δpost| <= 1` (360 Hz 정수 sample,
  round-half-to-even). 2차 점수·거리 선호·라벨 선호·record별 벌점은 **없다**.
- **CERTIFIED = 모든 maximum-cardinality monotone matching 에 공통으로 든 간선**.
  최적 경로에 따라 달라지는 간선은 `AMBIGUOUS` 이고 unmatched 로 남는다.
  forced edge 는 prefix/suffix DP 로 판정한다 — 최적 매칭을 열거하지 않는다.

In [ ]:
# ── 셀 9: Leg 2 규칙 확인 + (실행 후) record별 결과 ─────────────────────────
print("tolerance      :", BJ.RR_TOLERANCE_SAMPLES, "sample @", BJ.FS, "Hz")
print("secondary score: 없음 · distance/label 선호: 없음 · record 벌점: 없음")
print("status 값      :", BJ.STATUSES)
print("reason 값      :", [r for r in BJ.REASONS if r])

if need_results("6. Leg 2 record-wise join"):
    rows = load_csv("record_class_coverage.csv")
    for row in rows[:50]:
        print(f"  {row['record']:>10}  {row['certified_coverage']}")

## 7. DS1 gate report

12개 gate 전부 통과해야 규칙이 자격을 얻는다. 90% pooled coverage 로는 부족하다는
Q5-B-0 의 교훈이 class·record 하위 꼬리 gate 로 남아 있다.

In [ ]:
# ── 셀 10: DS1 gate 표 — decision.json 에서 읽는다 ──────────────────────────
print("등록 임계값:")
for name, value in (("3 overall coverage", BJ.GATE_COVERAGE_MIN),
                    ("4 S coverage", BJ.GATE_S_COVERAGE_MIN),
                    ("5 per-class coverage", BJ.GATE_PER_CLASS_COVERAGE_MIN),
                    ("6 class balance", BJ.GATE_CLASS_BALANCE_MIN),
                    ("7 record coverage", BJ.GATE_RECORD_COVERAGE_MIN),
                    ("7 record balance", BJ.GATE_RECORD_BALANCE_MIN),
                    ("8 agreement overall", BJ.GATE_AGREEMENT_OVERALL_MIN),
                    ("8 agreement per class", BJ.GATE_AGREEMENT_PER_CLASS_MIN),
                    ("10 signal/null", BJ.GATE_SIGNAL_TO_NULL_MIN),
                    ("12 S share inflation", BJ.GATE_S_SHARE_INFLATION_MAX)):
    print(f"  {name:<24} {value}")

if need_results("7. DS1 gate report"):
    decision = load_json("decision.json")
    print(f"\n{'gate':<28} {'pass':<6} value / threshold")
    for gate in decision["gates"]:
        print(f"  {gate['gate']:<26} {str(gate['passed']):<6} "
              f"{gate['value']} / {gate['threshold']}")
    print("\n통과:", decision["gates_passed"], "/", decision["gates_total"])

## 8. DS2 frozen gate report

DS1 규칙·source hash·테스트·환경·임계값·DS1 보고서가 **동결된 뒤에만** 같은 지지
gate(2-8, 12)를 DS2 에 **한 번** 적용한다. DS2 는 null 을 다시 돌리지 않고 어떤
상수도 바꾸지 않는다. **DS2 gate 가 실패하면 `JOIN_SELECTION_BIASED` 로 멈추고
V10 probability 를 열지 않는다.**

DS2 per-beat class label 은 이 단계 전까지 봉인이며, **join 규칙 선택에는 절대
쓰이지 않는다.**

In [ ]:
# ── 셀 11: DS2 지지 gate (동결 후 1회) ──────────────────────────────────────
print("DS2 에 적용되는 gate: 2-8 과 12 (null 재실행 없음, 상수 변경 없음)")
print("DS2 label 봉인 해제는 별도 토큰이 필요하다:", BJ.DS2_LABEL_RELEASE_FLAG)

if need_results("8. DS2 frozen gate report"):
    decision = load_json("decision.json")
    ds2 = [g for g in decision["gates"] if g.get("detail", "").startswith("DS2")]
    for gate in ds2 or decision["gates"]:
        print(f"  {gate['gate']:<26} passed={gate['passed']} value={gate['value']}")

## 9. negative-control / null plots

세 음성대조군은 각각 **완전한 Leg 2 를 다시 돌린다** — 후보 간선 구성, record별
최대 매칭, certification, 감사 통계 전부. `SEQUENCE_RELATIONSHIP` 외에는 아무것도
바뀌지 않는다.

`J_null_max[b] = max(J_wrong[b], J_shuffle[b], J_shift[b])` · master seed `2026017` ·
10,000 replicate. bootstrap 은 record-cluster 2,000 replicate, seed `2026018`.

**규칙이 바뀌면 저장된 null 을 재사용할 수 없다** — `rule_fingerprint` 가 함께
움직이고 `assert_null_matches_rule()` 이 거부한다.

In [ ]:
# ── 셀 12: TRUE J_min 대 max-null 분포 ──────────────────────────────────────
print("families:", BJ.CONTROL_FAMILIES)
print("seeds   :", BJ.MASTER_SEED, "/", BJ.BOOTSTRAP_SEED)
print("replicates:", BJ.N_NULL_REPLICATES, "/", BJ.N_BOOTSTRAP_REPLICATES)

if need_results("9. negative-control / null"):
    import matplotlib.pyplot as plt
    null = load_json("null_summary.json")
    BJ.assert_null_matches_rule(null)          # 완화된 규칙이 물려받지 못하게
    boot = load_json("bootstrap.json")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(null["j_null_max"], bins=60, color="#9aa5b1",
            label="max-null $J_{null,max}$")
    ax.axvline(null["q95"], color="#e08a00", ls="--", label=f"q95 {null['q95']:.4f}")
    ax.axvline(null["q99"], color="#c23b22", ls="--", label=f"q99 {null['q99']:.4f}")
    ax.axvline(null["j_true"], color="#1f6feb", lw=2,
               label=f"TRUE $J_{{min}}$ {null['j_true']:.4f}")
    ax.set_xlabel("$J_{min}$"); ax.set_ylabel("replicates")
    ax.set_title("TRUE $J_{min}$ vs family-wise max-null")
    ax.legend(); fig.tight_layout(); plt.show()

    print("signal_to_null:", null["signal_to_null"],
          ">= ", BJ.GATE_SIGNAL_TO_NULL_MIN)
    print("bootstrap 95% CI of (J_min_TRUE - q95):",
          boot["ci_low"], boot["ci_high"])

## 10. class·record coverage plots

`processed` 는 언제나 **V9/V10 위치 행**이지 mamba 행이 아니다. coverage 의 분모는
그 행이다.

In [ ]:
# ── 셀 13: record별 certified coverage · N/S/V class coverage · balance ─────
if need_results("10. coverage plots"):
    import matplotlib.pyplot as plt
    rows = load_csv("record_class_coverage.csv")
    per_record = [(r["record"], float(r["certified_coverage"]))
                  for r in rows if not r["record"].startswith("__class_")]
    per_class = [(r["record"].replace("__class_", ""),
                  float(r["certified_coverage"]))
                 for r in rows if r["record"].startswith("__class_")]
    decision = load_json("decision.json")
    gates = {g["gate"]: g for g in decision["gates"]}

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2),
                             gridspec_kw={"width_ratios": [3, 1, 1]})
    axes[0].bar([r for r, _v in per_record], [v for _r, v in per_record],
                color="#1f6feb")
    axes[0].axhline(BJ.GATE_RECORD_COVERAGE_MIN, color="#c23b22", ls="--",
                    label=f"record floor {BJ.GATE_RECORD_COVERAGE_MIN}")
    axes[0].set_title("record별 certified coverage")
    axes[0].tick_params(axis="x", rotation=90); axes[0].legend()

    axes[1].bar([c for c, _v in per_class], [v for _c, v in per_class],
                color=["#5a7d9a", "#e08a00", "#7a5195"])
    axes[1].axhline(BJ.GATE_PER_CLASS_COVERAGE_MIN, color="#c23b22", ls="--")
    axes[1].set_title("N/S/V class coverage")

    balance = [("class_coverage_balance", gates["6_class_coverage_balance"]["value"]),
               ("record_coverage_balance",
                gates["7_record_coverage"]["value"]["balance"])]
    axes[2].bar([b for b, _v in balance], [v for _b, v in balance],
                color="#2f7d32")
    axes[2].axhline(BJ.GATE_CLASS_BALANCE_MIN, color="#c23b22", ls="--")
    axes[2].set_title("balance gates"); axes[2].tick_params(axis="x", rotation=20)
    fig.tight_layout(); plt.show()

## 11. ambiguous / unmatched 원인표

`AMBIGUOUS` 는 **실패가 아니라 정직한 보고**다. 여러 최적 경로에 걸쳐 달라지는
간선은 certify 하지 않고 unmatched 로 남긴다. 반복 RR 때문에 너무 많이 남으면
그것이 `JOIN_UNRESOLVED` — "식별 가능한 것이 없다" 는 유효한 결과다.

Leg 1 실패와 Leg 2 실패는 **분리해서** 센다.

In [ ]:
# ── 셀 14: 원인별 개수 + Leg 1 / Leg 2 실패 분해 ───────────────────────────
if need_results("11. ambiguous/unmatched 원인표"):
    import collections
    import matplotlib.pyplot as plt
    rows = load_csv("unmatched_and_ambiguous.csv")
    reasons = collections.Counter(r["drop_or_unmatched_reason"] for r in rows)
    legs = collections.Counter(r["failed_leg"] or "none" for r in rows)

    print(f"{'reason':<40} count")
    for reason, count in reasons.most_common():
        print(f"  {reason:<38} {count}")
    print()
    print("leg별 실패 수:", dict(legs))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].bar(list(reasons), list(reasons.values()), color="#8a6d3b")
    axes[0].set_title("ambiguity / unmatched reason counts")
    axes[0].tick_params(axis="x", rotation=30)
    axes[1].bar(list(legs), list(legs.values()), color="#c23b22")
    axes[1].set_title("Leg 1 · Leg 2 별 실패 수")
    fig.tight_layout(); plt.show()

## 12. equal-count 36 vs mismatch 8 진단 비교

**진단용 층일 뿐이다.** 어느 층도 제외하거나 다른 matcher 를 주거나 실패한 primary
gate 를 구제하는 데 쓸 수 없다. 개수가 같다는 것은 위치 동일성이 아니다 — mamba 는
주석 위치 `pos`, V9/V10 은 검출 위치 `p` 로 경계를 자르므로 drop-one/add-one 상쇄가
가능하다.

In [ ]:
# ── 셀 15: 두 층 비교 ───────────────────────────────────────────────────────
ledger = BJ.build_ledger()
mismatched = [(s, r.record, r.delta) for s in BJ.SPLITS for r in ledger[s]
              if r.stratum == BJ.STRATUM_MISMATCH]
print("사전 등록된 불일치 record:", mismatched)
print("두 층 모두 동일한 matcher 를 통과한다.")

if need_results("12. equal-count 36 vs mismatch 8"):
    import matplotlib.pyplot as plt
    strata = load_json("decision.json").get("strata", {})
    if strata:
        names = list(strata)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(names, [strata[n]["coverage"] for n in names],
               color=["#1f6feb", "#e08a00"])
        ax.axhline(BJ.GATE_COVERAGE_MIN, color="#c23b22", ls="--")
        ax.set_title("equal-count 36 vs mismatch 8 — certified coverage")
        fig.tight_layout(); plt.show()
        for name in names:
            print(f"  {name:<18} {strata[name]}")

## 13. record 105 · 111 · 116 · 208 · 222 상세표

개수가 어긋나는 record 들이다. **전역 정렬이 왜 금지인지**를 보여 주는 자리이기도
하다 — DS2 의 105·111·222 결손은 전역 정렬에서 이후 모든 record 를 밀어 버린다.

`222` 는 자격검증에서 PPV 0.4873 으로 per-record floor 를 못 넘겼지만(주석 1,257 대
검출 2,477 → PPV 상한 0.5075) **primary 에서 제외·가중·보정하지 않는다.** 낮은
전문가-참조 PPV 가 올바른 beat join 을 실패시키게 두지 않고, 좋은 beat join 으로
미주석 P 검출의 생물학적 의미를 결론짓지도 않는다.

In [ ]:
# ── 셀 16: 불일치 record 상세 ───────────────────────────────────────────────
WATCH = ("105", "111", "116", "208", "222")
print(f"{'split':<5} {'rec':>4} {'cache_n':>8} {'mamba_n':>8} {'diff':>5} "
      f"{'cache@':>8} {'mamba@':>8}")
for split in BJ.SPLITS:
    for row in BJ.build_ledger()[split]:
        if row.record in WATCH:
            print(f"{split:<5} {row.record:>4} {row.cache_n:>8} {row.mamba_n:>8} "
                  f"{row.delta:>5} {row.cache_start:>8} {row.mamba_start:>8}")

if need_results("13. record 상세표"):
    rows = load_csv("record_class_coverage.csv")
    for row in rows:
        if row["record"] in WATCH:
            print(f"  {row['record']}  coverage={row['certified_coverage']}")

## 14. record 232 — S-share 원분포 대 certification 후 inflation

**join 이전에 이미** DS2 S beat 1,837 중 record `232` 가 1,382 개, **75.2%** 다.
이것은 **source concentration** 이고, 유리한 join 부분집합을 골라도 고쳐지지 않는다.

join gate 12 의 `S_share_inflation` 은 **certification 이 그 편중을 더 키웠는지**만
본다(≤ 1.25). parent spec 의 **절대 50% ceiling 을 완화하거나 대체하지 않는다.**

→ **join 이 성공해도 parent association 은 자기 gate 때문에 막힐 수 있다.** 그
해소는 별도 parent-spec 개정이 필요하고, 이 join 설계는 그런 개정을 하지 않는다.

In [ ]:
# ── 셀 17: 232 의 원 share 대 certified share ──────────────────────────────
print(f"source: record 232 = {BJ.RECORD_232_S_BEATS}/{BJ.DS2_S_BEATS_TOTAL} "
      f"DS2 S beats = {BJ.RECORD_232_S_SHARE:.4f}")
print(f"parent 절대 ceiling : {BJ.PARENT_ABSOLUTE_RECORD_S_SHARE_CEILING} "
      f"→ 이미 초과 (join 이전부터)")
print(f"join gate 12 ceiling: inflation <= {BJ.GATE_S_SHARE_INFLATION_MAX} "
      f"(source 대비 비율, 절대 share 가 아니다)")

if need_results("14. record 232"):
    import matplotlib.pyplot as plt
    shares = load_json("decision.json").get("s_share", {})
    if shares:
        records = sorted(shares)
        fig, ax = plt.subplots(figsize=(9, 4))
        width = 0.4
        idx = range(len(records))
        ax.bar([i - width / 2 for i in idx],
               [shares[r]["source_share"] for r in records], width,
               label="source share", color="#9aa5b1")
        ax.bar([i + width / 2 for i in idx],
               [shares[r]["certified_share"] for r in records], width,
               label="certified share", color="#1f6feb")
        ax.set_xticks(list(idx)); ax.set_xticklabels(records, rotation=90)
        ax.set_title("record 232 포함 — S source share vs certified share")
        ax.legend(); fig.tight_layout(); plt.show()
        print("232 inflation:", shares.get("232", {}).get("inflation"))

## 15. decision tree 최종 판정

first-failure-wins 로 **하나의 primary decision** 만 기록한다. 그러나 모든 audit
gate 의 수치와 pass/fail 은 따로 전부 저장한다 — 첫 실패 뒤에 통과한 gate 도
남는다.

In [ ]:
# ── 셀 18: 최종 판정 ────────────────────────────────────────────────────────
print("판정 후보:", BJ.DECISIONS)
print()
if not need_results("15. decision tree"):
    print(json.dumps(BJ.not_run_decision("implementation only; execution "
                                         "awaiting the separate approval"),
                     indent=2, ensure_ascii=False))
else:
    decision = load_json("decision.json")
    print("decision              :", decision["decision"])
    print("first_stopping_reason :", decision["first_stopping_reason"])
    print("failed_leg            :", decision["failed_leg"])
    print("gates                 :", decision["gates_passed"], "/",
          decision["gates_total"])
    for flag in ("training_performed", "model_scored",
                 "v10_probability_opened", "association_performed"):
        print(f"  {flag:<24}", decision.get(flag))

## 16. 사람이 읽을 수 있는 해석 요약

이 절은 `summary.md` 를 그대로 보여 준다. **숫자를 여기에 옮겨 적지 않는다.**

읽을 때 함께 기억할 것:

- `JOIN_IDENTIFIABLE` 은 **beat identity map 을 만들어도 된다**는 뜻일 뿐이다.
  P-timing association 이 아니고, V10 probability 를 여는 승인도 아니다.
- `JOIN_UNRESOLVED` 는 실패가 아니라 **"현재 산출물로는 식별 불가"** 라는 유효한
  결과다. Q5-B-0 이 남긴 교훈대로, 많이 붙었다와 쓸 수 있다는 다르다.
- 자격검증은 per-record floor **정확히 5/6** 으로 통과했다 — 여유가 0이었다.
  join 이 이것을 "측정 품질이 균일하다" 는 증거로 승격시킬 수 없다.

In [ ]:
# ── 셀 19: summary.md 그대로 출력 ──────────────────────────────────────────
if need_results("16. 해석 요약"):
    with open(bundle_path("summary.md"), encoding="utf-8") as fh:
        print(fh.read())

## 17. Drive bundle 저장 및 ingest 단계

**이 notebook 은 run bundle 을 만들지 않는다.** 실제 실행 승인 이후에만 아래 12개
파일이 생기고, 기존 자산을 덮어쓰지 않는다. `join_map` 에는 V10 probability 값을
저장하지 않는다.

실행 후 순서: ① Drive 에 bundle 저장 → ② 실행된 notebook 커밋 →
③ `python pipelines/ingest_run.py --results result.json --notebook notebooks/…ipynb`
로 실행 로그 카드 생성(수치는 실측, LLM 이 지어내지 않는다).

In [ ]:
# ── 셀 20: bundle 계약 확인 (생성하지 않는다) ──────────────────────────────
print("run 디렉터리:", f"MyDrive/{BJ.DRIVE_RUN_REL}/<timestamp>_{BJ.RUN_DIR_SUFFIX}/")
print("필수 파일:")
for name in BJ.BUNDLE_FILES:
    print("  -", name)
print()
print("join_map 열:", BJ.JOIN_MAP_FIELDS)
print("join_map 금지 열:", BJ.JOIN_MAP_BANNED_FIELDS)
print()
if have_results():
    complete, missing = BJ.bundle_is_complete(globals().get("RUN_DIR", ""))
    print("bundle 완전:", complete, "· 누락:", missing)
else:
    print("지금은 bundle 이 없다 — IMPLEMENTED / FULL RESULT NOT RUN.")
    print("Drive 변경 0건 · 확률 NPZ 열람 0건 · DS2 label 열람 0건.")